# 📡 AIOps — Session 15
# Kubernetes Telemetry and Observability

---

| | |
|---|---|
| **Course** | AIOps Engineering |
| **Lab** | Lab 15 |
| **Topics** | Metrics · Logs · Events · Prometheus Concepts · Grafana Concepts · Health Signals |
| **Duration** | ~90 minutes |
| **Platform** | Kubernetes (kubectl must be available in this JupyterLab environment) |

---

## 🎯 Learning Objectives

By completing this lab you will be able to:

1. **Explain** the three pillars of Kubernetes observability: metrics, logs, and events
2. **Collect** real metrics, logs, and events from a live K8s cluster using `kubectl`
3. **Understand** how Prometheus scrapes and stores time-series metrics
4. **Build** Prometheus-style PromQL queries and interpret their results
5. **Design** Grafana-style dashboard panels and alert rules
6. **Identify** Kubernetes health signals — OOMKill, CrashLoopBackOff, resource pressure
7. **Implement** a multi-signal health scoring system that ranks services by operational risk

---

## 🗺️ Lab Architecture

We deploy a **media-streaming platform** on Kubernetes:

```
  [cdn-gateway] ──▶ [stream-service]  ──▶ [encoding-service]  ──▶ [storage-service]
                ├──▶ [auth-service]   ──▶ [token-db]
                └──▶ [recommend-svc]  ──▶ [ml-engine]
                                      └──▶ [metrics-db]
```

---

## 📋 Lab Phases

| Phase | Title | Key Concept |
|-------|-------|-------------|
| **0** | Environment Setup | Libraries + kubectl |
| **1** | Deploy Instrumented Services | K8s manifests with annotations |
| **2** | The Three Pillars: Metrics | Prometheus data model + scraping |
| **3** | The Three Pillars: Logs | Structured logging + log levels |
| **4** | The Three Pillars: Events | K8s event taxonomy |
| **5** | Prometheus Concepts & PromQL | Counters, Gauges, Histograms, PromQL |
| **6** | Grafana Dashboard Design | Panel types, alert rules, thresholds |
| **7** | Kubernetes Health Signals | Restarts, OOMKill, Pressure, Saturation |
| **8** | Multi-Signal Health Scoring | Composite risk ranking |
| **9** | Cleanup | `kubectl delete` |

> ▶️ **Run each cell in order using `Shift + Enter`**

---
## Phase 0 — Environment Setup

**Purpose:** Install all Python libraries and verify `kubectl` connectivity.

### Libraries we need

| Library | Purpose |
|---------|--------|
| `pandas`, `numpy` | Data manipulation and time-series modelling |
| `matplotlib`, `seaborn` | Grafana-style dashboard panels and heatmaps |
| `networkx` | Service topology graph |
| `tabulate` | Formatted console tables |

> ⏱️ Takes ~60 seconds on first run.

In [ ]:
# ── Install required libraries ─────────────────────────────────────────────────
import sys
!{sys.executable} -m pip install -q pandas numpy matplotlib seaborn networkx scipy tabulate
print("✅ Libraries installed")

### Imports and global settings

All imports are centralised here. `random.seed(42)` ensures every synthetic
metric, log, and event is fully reproducible across re-runs.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import subprocess, json, time, random, math
from datetime import datetime, timedelta
from collections import defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import networkx as nx
from tabulate import tabulate

random.seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.facecolor':'#f8f9fa','axes.facecolor':'#ffffff',
                      'axes.grid':True,'grid.alpha':0.3,'font.size':10})
LAB_START = datetime.now()
print("✅ Imports ready")
print(f"   Lab started at: {LAB_START.strftime('%Y-%m-%d %H:%M:%S')}")

### Verify kubectl connectivity

`kubectl` must reach the cluster before deployment.
If this fails, check your `KUBECONFIG` environment variable.

In [ ]:
# ── Verify kubectl access ──────────────────────────────────────────────────────
def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.returncode != 0 and result.stderr.strip(): print('STDERR:', result.stderr.strip())
    return result.stdout.strip(), result.returncode

print("=" * 60)
print("kubectl version (client):")
run("kubectl version --client --short 2>/dev/null || kubectl version --client")
print("\nCluster nodes:")
out, rc = run("kubectl get nodes")
if rc == 0:
    print("\n✅ kubectl connected — ready to deploy!")
else:
    print("\n❌ kubectl cannot reach the cluster. Fix KUBECONFIG before continuing.")

---
## Phase 1 — Deploy Instrumented Services on Kubernetes

**Purpose:** Create a realistic microservices platform whose pods produce rich
telemetry across all three observability pillars.

### Why instrumentation annotations matter

Prometheus discovers which pods to scrape by reading **pod annotations**:

```yaml
annotations:
  prometheus.io/scrape: 'true'
  prometheus.io/port:   '8080'
  prometheus.io/path:   '/metrics'
```

Without these annotations, Prometheus ignores the pod entirely.

### Service topology

```
cdn-gateway ──▶ stream-service  ──▶ encoding-service ──▶ storage-service
            ├──▶ auth-service    ──▶ token-db
            └──▶ recommend-svc  ──▶ ml-engine
                                └──▶ metrics-db
```

> ⏱️ Allow 30–60 seconds for all pods to reach `Running` state.

In [ ]:
# ── Define services, dependency graph, and create namespace ───────────────────
NAMESPACE = 'aiops-lab15'

SERVICES = [
    ('cdn-gateway',       8080, 2, 'gateway',  '50m',  '64Mi'),
    ('stream-service',    8081, 2, 'backend',  '100m', '128Mi'),
    ('auth-service',      8082, 2, 'backend',  '50m',  '64Mi'),
    ('recommend-svc',     8083, 1, 'backend',  '200m', '256Mi'),
    ('encoding-service',  8084, 1, 'backend',  '500m', '512Mi'),
    ('ml-engine',         8085, 1, 'backend',  '300m', '512Mi'),
    ('storage-service',   8086, 1, 'backend',  '50m',  '128Mi'),
    ('token-db',          5432, 1, 'database', '50m',  '128Mi'),
    ('metrics-db',        5433, 1, 'database', '50m',  '128Mi'),
]

DEPENDENCY_GRAPH = {
    'cdn-gateway':      ['stream-service', 'auth-service', 'recommend-svc'],
    'stream-service':   ['encoding-service'],
    'encoding-service': ['storage-service'],
    'auth-service':     ['token-db'],
    'recommend-svc':    ['ml-engine', 'metrics-db'],
    'ml-engine':        [], 'storage-service': [], 'token-db': [], 'metrics-db': [],
}

ns_yaml = ('apiVersion: v1\nkind: Namespace\nmetadata:\n'
           f'  name: {NAMESPACE}\n'
           '  labels:\n    lab: session15\n    managed-by: aiops-lab\n')
with open('/tmp/lab15-ns.yaml', 'w') as f: f.write(ns_yaml)
run(f'kubectl apply -f /tmp/lab15-ns.yaml')
print(f'\n✅ Namespace "{NAMESPACE}" ready')

### Kubernetes manifests with full observability annotations

Each deployment is configured with:
- **Prometheus scrape annotations** — enables automatic metric discovery
- **Resource requests and limits** — required for memory saturation health signals
- **Liveness and readiness probes** — generates the K8s events we examine in Phase 4

In [ ]:
# ── Build manifests using string list (avoids f-string nesting issues) ─────────
def build_manifest(name, port, replicas, tier, cpu_req, mem_req):
    cpu_limit = str(int(cpu_req.replace('m','')) * 4) + 'm'
    mem_limit = str(int(mem_req.replace('Mi','')) * 2) + 'Mi'
    lines = [
        '---', 'apiVersion: apps/v1', 'kind: Deployment', 'metadata:',
        f'  name: {name}', f'  namespace: {NAMESPACE}', '  labels:',
        f'    app: {name}', f'    tier: {tier}', 'spec:', f'  replicas: {replicas}',
        '  selector:', '    matchLabels:', f'      app: {name}',
        '  template:', '    metadata:', '      labels:', f'        app: {name}',
        '      annotations:',
        '        prometheus.io/scrape: "true"',
        f'        prometheus.io/port: "{port}"',
        '        prometheus.io/path: "/metrics"',
        '    spec:', '      containers:', f'      - name: {name}',
        '        image: nginx:alpine', '        ports:', '        - containerPort: 80',
        '        resources:', '          requests:',
        f'            cpu: "{cpu_req}"', f'            memory: "{mem_req}"',
        '          limits:', f'            cpu: "{cpu_limit}"', f'            memory: "{mem_limit}"',
        '        livenessProbe:', '          httpGet: {path: /, port: 80}',
        '          initialDelaySeconds: 5', '          periodSeconds: 10',
        '        readinessProbe:', '          httpGet: {path: /, port: 80}',
        '          initialDelaySeconds: 3', '          periodSeconds: 5',
        '---', 'apiVersion: v1', 'kind: Service', 'metadata:',
        f'  name: {name}', f'  namespace: {NAMESPACE}',
        'spec:', '  selector:', f'    app: {name}',
        '  ports:', f'  - port: {port}', '    targetPort: 80',
    ]
    return '\n'.join(lines) + '\n'

manifest_all = ''.join(build_manifest(*s) for s in SERVICES)
with open('/tmp/lab15-services.yaml', 'w') as f: f.write(manifest_all)
run(f'kubectl apply -f /tmp/lab15-services.yaml')
print(f'\n✅ {len(SERVICES)} services applied')

### Wait for pods to become ready

All pods must reach `1/1 Ready` before collecting telemetry.
The readiness probe result appears in K8s events — we examine those in Phase 4.

In [ ]:
# ── Wait for rollouts and verify pod health ────────────────────────────────────
print("⏳ Waiting for all deployments to roll out...")
for name, *_ in SERVICES:
    run(f'kubectl rollout status deployment/{name} -n {NAMESPACE} --timeout=90s')
print("\n" + "=" * 60)
print("Pod Status:")
run(f'kubectl get pods -n {NAMESPACE} -o wide')
print("\nServices:")
run(f'kubectl get svc -n {NAMESPACE}')
print("\n✅ All services deployed — ready to collect telemetry")

---
## Phase 2 — The First Pillar: Metrics 📊

### 📖 Concept: What Are Kubernetes Metrics?

> **Metrics** are **numeric measurements** collected at regular intervals.
> They are the fastest, cheapest observability signal — ideal for alerting and dashboards.

Kubernetes metrics come from three layers:

| Layer | What it measures | Tool |
|-------|-----------------|------|
| **Infrastructure** | Node CPU, memory, disk, network | `kubelet`, `node-exporter` |
| **Platform** | Pod CPU, memory, restarts | `kube-state-metrics`, `cAdvisor` |
| **Application** | Request rate, latency, error rate | Your app's `/metrics` endpoint |

### The Prometheus Data Model

Prometheus stores every metric as a **time series** identified by:
- A **metric name** — e.g., `http_requests_total`
- A set of **labels** — e.g., `{service="cdn-gateway", status="200"}`
- A sequence of **(timestamp, value)** pairs

### Prometheus Metric Types

| Type | Description | Example |
|------|-------------|--------|
| **Counter** | Monotonically increasing, never decreases | `http_requests_total` |
| **Gauge** | Can go up or down | `memory_usage_bytes` |
| **Histogram** | Samples into buckets | `http_request_duration_seconds` |
| **Summary** | Client-side quantiles | `rpc_duration_summary` |

### Step 2a — Collect live resource metrics with kubectl top

`kubectl top` reads real-time CPU and memory from the **metrics-server**.
These are **Gauge** metrics — they change continuously as workloads run.

In [ ]:
# ── Collect live resource metrics ─────────────────────────────────────────────
print("📊 Live Pod Resource Metrics (from metrics-server):")
print("=" * 60)
out_pods, rc_pods = run(f'kubectl top pods -n {NAMESPACE} --no-headers 2>/dev/null')
print("\n📊 Live Node Resource Metrics:")
print("=" * 60)
run('kubectl top nodes --no-headers 2>/dev/null')
if rc_pods != 0 or not out_pods.strip():
    print("\n⚠️  metrics-server not available — continuing with synthetic metrics.")

### Step 2b — Synthesise a Prometheus metrics time-series

We generate **60 minutes of realistic Prometheus-format metrics** for all services.

The traffic model includes:
- A realistic **lunch-hour spike** at T=20–40 min (all services)
- Gradual **degradation** after T=30 min for `recommend-svc` and `ml-engine`

These two services represent the **problem cases** we will detect and score in Phases 7–8.

In [ ]:
# ── Synthesise 60 minutes of Prometheus-format time-series metrics ─────────────
SCRAPE_INTERVAL = 15
DURATION_MINS   = 60
N_POINTS        = int(DURATION_MINS * 60 / SCRAPE_INTERVAL)
TIMESTAMPS      = [LAB_START + timedelta(seconds=i*SCRAPE_INTERVAL) for i in range(N_POINTS)]
T_MINUTES       = [i*SCRAPE_INTERVAL/60 for i in range(N_POINTS)]

def traffic_profile(t_min, base_rps, degraded=False):
    spike   = 1.8 * np.exp(-0.5 * ((t_min - 30) / 8)**2)
    degrade = -0.3 * base_rps * (t_min / DURATION_MINS) if degraded else 0
    return max(0.1, base_rps + spike + np.random.normal(0, 0.05*base_rps) + degrade)

SVC_CONFIG = {
    'cdn-gateway':      {'base_rps':850,  'p99_ms':45,   'mem_mb':180,  'degraded':False},
    'stream-service':   {'base_rps':420,  'p99_ms':120,  'mem_mb':310,  'degraded':False},
    'auth-service':     {'base_rps':390,  'p99_ms':35,   'mem_mb':95,   'degraded':False},
    'recommend-svc':    {'base_rps':210,  'p99_ms':280,  'mem_mb':490,  'degraded':True},
    'encoding-service': {'base_rps':85,   'p99_ms':950,  'mem_mb':820,  'degraded':False},
    'ml-engine':        {'base_rps':95,   'p99_ms':1200, 'mem_mb':950,  'degraded':True},
    'storage-service':  {'base_rps':120,  'p99_ms':60,   'mem_mb':220,  'degraded':False},
    'token-db':         {'base_rps':380,  'p99_ms':18,   'mem_mb':380,  'degraded':False},
    'metrics-db':       {'base_rps':60,   'p99_ms':25,   'mem_mb':290,  'degraded':False},
}

metrics_rows = []
req_ctrs = {s:0 for s in SVC_CONFIG}
err_ctrs = {s:0 for s in SVC_CONFIG}

for i, (ts, t_min) in enumerate(zip(TIMESTAMPS, T_MINUTES)):
    for svc, cfg in SVC_CONFIG.items():
        rps  = traffic_profile(t_min, cfg['base_rps'], cfg['degraded'])
        req_ctrs[svc] += rps * SCRAPE_INTERVAL
        p99  = cfg['p99_ms'] * (rps/cfg['base_rps']) + np.random.normal(0, 10)
        err  = 0.002 + (0.08 if cfg['degraded'] and t_min > 30 else 0) * (rps/cfg['base_rps'])
        err  = min(max(err + np.random.uniform(0, 0.005), 0), 0.95)
        err_ctrs[svc] += rps * SCRAPE_INTERVAL * err
        mem  = cfg['mem_mb']*1024**2 * (1 + 0.001*i + np.random.normal(0, 0.02))
        metrics_rows.append({
            'timestamp': ts, 't_min': round(t_min,2), 'service': svc,
            'http_requests_total': int(req_ctrs[svc]),
            'http_requests_per_second': round(rps, 2),
            'http_request_duration_p99_ms': round(max(1, p99), 2),
            'http_errors_total': int(err_ctrs[svc]),
            'error_rate': round(err, 5),
            'process_resident_memory_bytes': int(max(0, mem)),
        })

df_metrics = pd.DataFrame(metrics_rows)
print(f'✅ Generated {len(df_metrics):,} metric data points')
print('\nPeak error rates per service:')
peak_err = df_metrics.groupby('service')['error_rate'].max().sort_values(ascending=False)
for svc, rate in peak_err.items():
    bar = '█' * int(rate*20)
    print(f'  {svc:<22} {bar:<22} {rate:.2%}')

### Step 2c — Visualise the metrics time-series

This replicates what **Prometheus + Grafana** would display.

Key observations:
- The traffic **spike at T=20–40 min** affects all services simultaneously
- `recommend-svc` and `ml-engine` (red/orange) show **rising error rates after T=30 min** — this is the degradation we injected
- `encoding-service` and `ml-engine` have the highest baseline P99 latency — they are CPU-intensive workloads

In [ ]:
# ── Visualise metrics time-series (Grafana-style panels) ──────────────────────
HIGHLIGHT = {'recommend-svc':'#d62728', 'ml-engine':'#ff7f0e'}
NORMAL    = '#9ecae1'

fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True)
fig.suptitle('📊 Prometheus Metrics Dashboard — Media Streaming Platform (60 min)',
             fontsize=13, fontweight='bold')

for svc in df_metrics['service'].unique():
    d   = df_metrics[df_metrics['service']==svc]
    col = HIGHLIGHT.get(svc, NORMAL)
    lw  = 2.2 if svc in HIGHLIGHT else 1.0
    axes[0].plot(d['t_min'], d['http_requests_per_second'],    color=col, lw=lw, label=svc)
    axes[1].plot(d['t_min'], d['error_rate']*100,              color=col, lw=lw)
    axes[2].plot(d['t_min'], d['http_request_duration_p99_ms'],color=col, lw=lw)

axes[0].set_ylabel('Requests / sec')
axes[0].set_title('① Request Rate (RPS) — Counter rate()', fontweight='bold', loc='left')
axes[0].legend(ncol=3, fontsize=8)
axes[1].set_ylabel('Error Rate (%)')
axes[1].set_title('② Error Rate — recommend-svc and ml-engine degrade after T=30min ⚠️',
                  fontweight='bold', loc='left', color='#d62728')
axes[1].axhline(5, color='red', linestyle='--', alpha=0.5, label='Alert threshold 5%')
axes[1].legend(fontsize=8)
axes[2].set_ylabel('P99 Latency (ms)')
axes[2].set_xlabel('Time (minutes)')
axes[2].set_title('③ P99 Latency — high-latency services: encoding-service, ml-engine',
                  fontweight='bold', loc='left')
axes[2].axhline(1000, color='orange', linestyle='--', alpha=0.5, label='SLO 1000ms')
axes[2].legend(fontsize=8)
for ax in axes:
    ax.axvspan(20, 40, alpha=0.05, color='blue')
    ax.text(30, ax.get_ylim()[1]*0.92, 'Traffic spike', ha='center', fontsize=7, color='blue')
plt.tight_layout()
plt.savefig('/tmp/lab15_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Metrics dashboard rendered")

---
## Phase 3 — The Second Pillar: Logs 📋

### 📖 Concept: What Are Kubernetes Logs?

> **Logs** are **discrete, timestamped text records** emitted by application processes.
> They provide the richest context about *what happened* — metrics tell you *that*
> something is wrong; logs tell you *why*.

In Kubernetes, every container writes to **stdout/stderr**. The kubelet buffers
these on the node. Log collectors like **Fluentd**, **Fluent Bit**, or **Loki**
ship them to a centralised store.

### Structured vs Unstructured Logging

| Style | Format | Searchability |
|-------|--------|--------------|
| **Unstructured** | Plain text | Poor — regex only |
| **Structured** | JSON / logfmt | Excellent — field-level queries |

Modern best practice is **structured JSON logging** with consistent fields.

### Log Level Severity Hierarchy

```
DEBUG  <  INFO  <  WARN  <  ERROR  <  FATAL
```

### Step 3a — Collect real container logs with kubectl

`kubectl logs` fetches the stdout/stderr stream from any running container.
In production these same lines flow to your log aggregator (Loki, Elasticsearch, Splunk).

In [ ]:
# ── Collect real container logs from running pods ──────────────────────────────
print("📋 Container logs from cdn-gateway (last 20 lines):")
print("-" * 60)
cmd = f'kubectl logs -n {NAMESPACE} deploy/cdn-gateway --tail=20 2>/dev/null'
out, rc = run(cmd)
if not out.strip():
    print("(nginx startup logs — no app traffic yet)")

print("\n📋 Previous container logs (crash/OOM logs if pod restarted):")
print("-" * 60)
cmd2 = f'kubectl logs -n {NAMESPACE} deploy/cdn-gateway --previous --tail=10 2>/dev/null'
out2, rc2 = run(cmd2)
if rc2 != 0 or not out2.strip():
    print("  (No previous logs — pod has not restarted yet)")

### Step 3b — Synthesise structured application logs

We generate realistic **structured JSON log lines** for each service.

Each line includes: `timestamp`, `level`, `service`, `pod_id`, `trace_id`, `message`.

Notice how the error messages form a **semantic cascade trail** matching the topology:
`ml-engine OOM` → `recommend-svc model failure` → `cdn-gateway upstream timeout`.
This trail is a primary signal for root cause analysis in downstream AIOps pipelines.

In [ ]:
# ── Synthesise structured JSON application logs ────────────────────────────────
LOG_TEMPLATES = {
    'cdn-gateway':      {'INFO':['GET /stream/{cid} 200 {lat}ms trace={tid}'],
                         'WARN':['Upstream stream-service slow: {lat}ms'],
                         'ERROR':['Upstream timeout {lat}ms: stream-service unreachable']},
    'stream-service':   {'INFO':['Stream started content={cid} codec=H264'],
                         'WARN':['Buffer fill rate low: {r}%'],
                         'ERROR':['Encoding stalled: job {tid} exceeded 30s']},
    'auth-service':     {'INFO':['Token validated user={uid} expires_in=3600s'],
                         'WARN':['Suspicious login user={uid} ip=10.{r}.{r}.1'],
                         'ERROR':['token-db pool exhausted waited {lat}ms']},
    'recommend-svc':    {'INFO':['Recommendations user={uid} latency={lat}ms'],
                         'WARN':['Model inference slow: {lat}ms > 500ms SLO'],
                         'ERROR':['ML model load failed: OOM during feature extraction']},
    'encoding-service': {'INFO':['Encoding job {tid} complete duration={lat}s'],
                         'WARN':['Queue depth: {r} jobs (threshold: 10)'],
                         'ERROR':['storage-service write timeout job {tid}']},
    'ml-engine':        {'INFO':['Inference batch size={r} latency={lat}ms'],
                         'WARN':['Memory pressure: RSS={r}MB approaching limit'],
                         'ERROR':['CUDA out of memory: batch_size={r}']},
    'storage-service':  {'INFO':['Object written key={tid} size={r}MB latency={lat}ms'],
                         'WARN':['Disk utilisation: {r}% threshold 80%'],
                         'ERROR':['Write failed: no space left on device']},
    'token-db':         {'INFO':['checkpoint complete {r} buffers written'],
                         'WARN':['long-running query {lat}ms on table tokens'],
                         'ERROR':['FATAL: connection pool exhausted']},
    'metrics-db':       {'INFO':['Query executed {lat}ms rows={r}'],
                         'WARN':['Table bloat: autovacuum recommended'],
                         'ERROR':['Connection pool exhausted: {r} clients waiting']},
}

DEGRADED_SVCS = {'recommend-svc', 'ml-engine'}
log_rows = []
for svc, templates in LOG_TEMPLATES.items():
    is_deg = svc in DEGRADED_SVCS
    lw = {'INFO':70,'WARN':20,'ERROR':10} if not is_deg else {'INFO':40,'WARN':30,'ERROR':30}
    tier   = next(s[3] for s in SERVICES if s[0]==svc)
    n_logs = {'gateway':400,'backend':200,'database':80}.get(tier,150)
    levels = list(lw.keys())
    wts    = [lw[l]/100 for l in levels]
    for _ in range(n_logs):
        ts    = LAB_START + timedelta(minutes=random.expovariate(0.3)*DURATION_MINS/10)
        level = random.choices(levels, weights=wts)[0]
        tmpl  = random.choice(templates[level])
        msg   = tmpl.format(cid=f'c{random.randint(10000,99999)}',
                             uid=f'u{random.randint(1000,9999)}',
                             tid=f'{random.randint(0xaaa,0xfff):x}{random.randint(0x1000,0xffff):x}',
                             lat=random.randint(5,2500), r=random.randint(10,99))
        log_rows.append({'timestamp':ts,'level':level,'service':svc,
                          'pod_id':f'{svc[:6]}-{random.randint(10000,99999)}',
                          'trace_id':f'{random.randint(0,0xffffffffffff):012x}',
                          'message':msg})

df_logs = pd.DataFrame(log_rows).sort_values('timestamp').reset_index(drop=True)
print(f'✅ Generated {len(df_logs):,} structured log lines')
summary = df_logs.groupby(['service','level']).size().unstack(fill_value=0)
print(tabulate(summary, headers='keys', tablefmt='rounded_outline'))
print('\n🔍 Sample ERROR records (JSON format):')
for _, row in df_logs[df_logs['level']=='ERROR'].head(3).iterrows():
    print(' ', json.dumps({'ts':row.timestamp.isoformat(),'level':row.level,
                            'svc':row.service,'trace':row.trace_id,'msg':row.message}))

### Step 3c — Log analysis: error rates and top error messages

Log analysis typically involves:
1. **Filtering** by severity level
2. **Aggregating** by time window to compute error rates
3. **Pattern mining** to find the most frequent error signatures

This is exactly what tools like **Kibana**, **Grafana Loki**, or **Splunk** automate.

In [ ]:
# ── Log analysis: level distribution and top error patterns ───────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('📋 Log Analysis Dashboard — Level Distribution & Error Patterns',
             fontsize=12, fontweight='bold')

pivot = df_logs.groupby(['service','level']).size().unstack(fill_value=0)
for col in ['INFO','WARN','ERROR']:
    if col not in pivot.columns: pivot[col] = 0
pivot = pivot[['INFO','WARN','ERROR']]
colors_lev = {'INFO':'#2196F3','WARN':'#FF9800','ERROR':'#F44336'}
bottom = np.zeros(len(pivot))
for level in ['INFO','WARN','ERROR']:
    ax1.bar(range(len(pivot)), pivot[level], bottom=bottom,
            label=level, color=colors_lev[level], alpha=0.85)
    bottom += pivot[level].values
ax1.set_xticks(range(len(pivot)))
ax1.set_xticklabels(pivot.index, rotation=35, ha='right', fontsize=8)
ax1.set_ylabel('Log line count')
ax1.set_title('Log Level Distribution per Service', fontweight='bold')
ax1.legend()

top_errors = df_logs[df_logs['level']=='ERROR']['message'].str[:55].value_counts().head(10)
colors_e = ['#d62728' if i<3 else '#ff7f0e' if i<6 else '#aaaaaa' for i in range(len(top_errors))]
ax2.barh(range(len(top_errors)), top_errors.values, color=colors_e)
ax2.set_yticks(range(len(top_errors)))
ax2.set_yticklabels([m[:50] for m in top_errors.index], fontsize=7)
ax2.invert_yaxis()
ax2.set_xlabel('Count')
ax2.set_title('Top 10 Error Message Patterns', fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/lab15_logs.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Log dashboard rendered")
print('\n📊 ERROR log % by service:')
err_rate = df_logs.groupby('service')['level'].apply(
    lambda s: (s=='ERROR').sum()/len(s)*100).sort_values(ascending=False)
for svc, rate in err_rate.items():
    print(f'  {svc:<22} {"▰"*int(rate/2):<20} {rate:.1f}%')

---
## Phase 4 — The Third Pillar: Events ⚡

### 📖 Concept: What Are Kubernetes Events?

> **Kubernetes events** are **discrete state-change notifications** emitted by the
> Kubernetes control plane. Unlike metrics (continuous) or logs (application-generated),
> events record specific things K8s *did* or *detected*.

Events are stored in etcd, accessible via `kubectl get events`, and retained for ~1 hour.

### Event Taxonomy

| Reason | Type | Meaning | Severity |
|--------|------|---------|----------|
| `Scheduled` / `Pulled` / `Started` | Normal | Healthy pod lifecycle | 🟢 Info |
| `Unhealthy` | Warning | Liveness or readiness probe failed | 🟡 Warning |
| `BackOff` | Warning | Container restart back-off | 🟡 Warning |
| `OOMKilling` | Warning | Container killed — exceeded memory limit | 🔴 Critical |
| `CrashLoopBackOff` | Warning | Container in persistent restart loop | 🔴 Critical |
| `FailedScheduling` | Warning | No node can fit the pod | 🔴 Critical |
| `Evicted` | Warning | Pod evicted due to node resource pressure | 🔴 Critical |

Events are the **ground truth** for infrastructure incidents — no inference needed.

### Step 4a — Collect real K8s events from the cluster

In a fresh deployment you will mainly see normal lifecycle events
(Scheduled, Pulled, Started) — exactly what a healthy service should produce.

In [ ]:
# ── Collect real Kubernetes events from the cluster ───────────────────────────
print(f'📡 Fetching K8s events from namespace: {NAMESPACE}')
events_raw, rc = run(f'kubectl get events -n {NAMESPACE} --sort-by=.lastTimestamp -o json 2>/dev/null')

k8s_events = []
if rc == 0 and events_raw.strip():
    try:
        ev_data = json.loads(events_raw)
        for item in ev_data.get('items', []):
            k8s_events.append({
                'time':   item.get('lastTimestamp',''),
                'type':   item.get('type',''),
                'reason': item.get('reason',''),
                'object': item.get('involvedObject',{}).get('name',''),
                'kind':   item.get('involvedObject',{}).get('kind',''),
                'msg':    item.get('message','')[:80],
            })
        print(f'✅ Retrieved {len(k8s_events)} real events')
    except json.JSONDecodeError:
        print('⚠️  Could not parse event JSON.')

if k8s_events:
    df_real = pd.DataFrame(k8s_events)
    print(tabulate(df_real[['reason','type','object','kind','msg']].head(15),
                   headers='keys', tablefmt='rounded_outline', showindex=False))
else:
    print('(No events yet — pods may still be starting)')

### Step 4b — Synthesise a realistic event stream

We generate simulated events modelling both **normal operations** and **incidents**:
an `ml-engine` OOM cycle and a `recommend-svc` degradation.

This synthetic stream feeds the health signal detector in Phase 7.

In [ ]:
# ── Synthesise K8s event stream ───────────────────────────────────────────────
T0 = LAB_START
all_svc_names = [s[0] for s in SERVICES]

# (reason, type, kind, svc_target, offset_s, message)
event_defs = [
    ('Scheduled',          'Normal',  'Pod',        '*',               2,    'Pod assigned to node-{n}'),
    ('Pulling',            'Normal',  'Pod',        '*',               4,    'Pulling image nginx:alpine'),
    ('Pulled',             'Normal',  'Pod',        '*',               8,    'Successfully pulled nginx:alpine in 1.2s'),
    ('Started',            'Normal',  'Pod',        '*',               10,   'Started container {svc}'),
    ('OOMKilling',         'Warning', 'Pod',        'ml-engine',       1820, 'Memory limit exceeded: killed process 1 limit=512Mi'),
    ('BackOff',            'Warning', 'Pod',        'ml-engine',       1835, 'Back-off restarting failed container ml-engine'),
    ('Unhealthy',          'Warning', 'Pod',        'ml-engine',       1850, 'Readiness probe failed: HTTP 503'),
    ('CrashLoopBackOff',   'Warning', 'Pod',        'ml-engine',       1900, 'Container ml-engine CrashLoopBackOff (4 restarts)'),
    ('Unhealthy',          'Warning', 'Pod',        'recommend-svc',   2100, 'Liveness probe failed: HTTP 503 from /health'),
    ('BackOff',            'Warning', 'Pod',        'recommend-svc',   2200, 'Back-off restarting failed container recommend-svc'),
    ('Unhealthy',          'Warning', 'Pod',        'encoding-service',2500, 'Readiness probe failed: encoding queue full'),
    ('NodeNotReady',       'Warning', 'Node',       None,              1950, 'Node node-2: KubeletNotReady — memory pressure'),
    ('EvictionThresholdMet','Warning','Node',       None,              1960, 'Eviction manager: memory.available < 200Mi on node-2'),
    ('ScalingReplicaSet',  'Normal',  'Deployment', 'cdn-gateway',     1200, 'Scaled up replica set cdn-gateway to 3'),
]

event_rows = []
for reason, etype, kind, svc_target, offset_s, msg_tmpl in event_defs:
    targets = all_svc_names if svc_target=='*' else ([svc_target] if svc_target else [None])
    for svc in targets:
        ts  = T0 + timedelta(seconds=offset_s + random.randint(-5,5))
        n   = random.randint(1,4)
        pod = f'{svc[:8]}-{random.randint(10000,99999)}' if svc else 'node-2'
        msg = msg_tmpl.format(svc=svc or '', n=n)
        event_rows.append({'timestamp':ts,'offset_s':offset_s,'type':etype,
                            'reason':reason,'kind':kind,'object':pod,
                            'service':svc,'message':msg})

df_events = pd.DataFrame(event_rows).sort_values('timestamp').reset_index(drop=True)
print(f'✅ Generated {len(df_events)} K8s events ({len(df_events[df_events["type"]=="Warning"])} warnings)')
warnings = df_events[df_events['type']=='Warning']
print(tabulate(warnings[['offset_s','reason','service','message']].head(12),
               headers=['T+s','Reason','Service','Message'],
               tablefmt='rounded_outline', showindex=False))

### Step 4c — Event timeline visualisation

`OOMKilling` and `CrashLoopBackOff` events (red ✕) require immediate investigation.
Normal lifecycle events (blue circles) confirm healthy pod startup.

This visualisation is available in Kubernetes dashboards like **Lens** and **K9s**.

In [ ]:
# ── K8s event timeline ─────────────────────────────────────────────────────────
EVENT_COLORS = {
    'OOMKilling':'#d62728','CrashLoopBackOff':'#d62728',
    'BackOff':'#ff7f0e','Unhealthy':'#ff7f0e','EvictionThresholdMet':'#ff7f0e',
    'NodeNotReady':'#9467bd','ScalingReplicaSet':'#2ca02c',
    'Scheduled':'#aec7e8','Pulled':'#aec7e8','Started':'#aec7e8','Pulling':'#c7c7c7',
}
YPOS = {svc:i for i,svc in enumerate(all_svc_names)}
YPOS[None] = len(all_svc_names)

fig, ax = plt.subplots(figsize=(15, 7))
for _, row in df_events.iterrows():
    x   = row['offset_s']/60
    y   = YPOS.get(row['service'], len(all_svc_names))
    col = EVENT_COLORS.get(row['reason'],'#888888')
    ax.scatter(x, y, c=col, s=180 if row['type']=='Warning' else 60,
               marker='X' if row['type']=='Warning' else 'o', alpha=0.85, zorder=3)

ax.set_yticks(range(len(all_svc_names)+1))
ax.set_yticklabels(all_svc_names + ['node / cluster'], fontsize=9)
ax.set_xlabel('Time (minutes)')
ax.set_title('⚡ Kubernetes Event Timeline — ✕ Warning  •  ● Normal', fontweight='bold')
legend_handles = [
    mpatches.Patch(color='#d62728',label='OOMKilling / CrashLoopBackOff (Critical)'),
    mpatches.Patch(color='#ff7f0e',label='Unhealthy / BackOff / Eviction (Warning)'),
    mpatches.Patch(color='#9467bd',label='NodeNotReady'),
    mpatches.Patch(color='#2ca02c',label='ScalingReplicaSet (Normal)'),
    mpatches.Patch(color='#aec7e8',label='Lifecycle events (Normal)'),
]
ax.legend(handles=legend_handles, fontsize=8, loc='upper right')
ax.set_xlim(-1, DURATION_MINS+1)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/lab15_events.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Event timeline rendered")

---
## Phase 5 — Prometheus Concepts & PromQL 🔎

### 📖 Concept: How Prometheus Works

```
Your Pod  ←── scrape /metrics every 15s ──  Prometheus Server (TSDB)
                                                  │  PromQL queries
                                             Grafana / AlertManager
```

Prometheus **pulls** (scrapes) metrics from targets. Each metric becomes a
**time series** — a sequence of (timestamp, value) pairs stored in the TSDB.

### PromQL — Prometheus Query Language

| Operation | PromQL syntax | Meaning |
|-----------|---------------|---------|
| Instant vector | `http_requests_total` | Current value of all matching series |
| Label filter | `http_requests_total{service="cdn-gateway"}` | Filter by label |
| Rate | `rate(http_requests_total[5m])` | Per-second rate over 5-min window |
| Increase | `increase(http_requests_total[1h])` | Total counter increase in 1 hour |
| Aggregation | `sum by (service) (rate(...))` | Sum across label dimensions |
| Alert expression | `rate(...) > 0.05` | Returns series where condition is true |

### Step 5a — Model the Prometheus TSDB structure

We build a Python representation of a Prometheus TSDB:
a dictionary of `metric_name{labels}` → `list of (timestamp, value)` pairs.
This is exactly how Prometheus stores data internally.

In [ ]:
# ── Model the Prometheus TSDB ─────────────────────────────────────────────────
prom_tsdb = defaultdict(list)
for _, row in df_metrics.iterrows():
    svc = row['service']
    ts  = row['timestamp']
    lb  = '{' + f'service="{svc}",namespace="{NAMESPACE}"' + '}'
    prom_tsdb[f'http_requests_total{lb}'].append(          (ts, row['http_requests_total']))
    prom_tsdb[f'http_request_duration_p99_s{lb}'].append(  (ts, row['http_request_duration_p99_ms']/1000))
    prom_tsdb[f'http_errors_total{lb}'].append(            (ts, row['http_errors_total']))
    prom_tsdb[f'process_resident_memory_bytes{lb}'].append((ts, row['process_resident_memory_bytes']))

print(f'✅ Prometheus TSDB modelled: {len(prom_tsdb)} unique time series')
print(f'   Each series has {N_POINTS} data points ({SCRAPE_INTERVAL}s scrape interval)')
sample = list(prom_tsdb.keys())[0]
print(f'\nSample series: {sample[:70]}...')
print('First 4 data points:')
for ts, val in prom_tsdb[sample][:4]:
    print(f'  {ts.strftime("%H:%M:%S")}  →  {val:.0f}')

### Step 5b — Implement PromQL query functions

We implement the three core PromQL functions in Python:
- `rate()` — per-second average rate of a counter
- `increase()` — total counter increase over a time window
- `promql_last()` — most recent value (instant vector)

These are the **exact computations** Prometheus runs when you execute a PromQL query.

In [ ]:
# ── Implement core PromQL functions ───────────────────────────────────────────
def promql_rate(pts, window_s=300):
    # rate(metric[5m]) — per-second average increase of a counter
    if len(pts) < 2: return 0.0
    t_end, v_end = pts[-1]
    for t_s, v_s in reversed(pts[:-1]):
        dt = (t_end - t_s).total_seconds()
        if dt >= window_s: return max(0.0, (v_end - v_s) / dt)
    dt = (pts[-1][0] - pts[0][0]).total_seconds()
    return max(0.0, (pts[-1][1] - pts[0][1]) / dt) if dt > 0 else 0.0

def promql_increase(pts, window_s=3600):
    # increase(metric[1h]) — total counter increase over window
    if len(pts) < 2: return 0.0
    t_end, v_end = pts[-1]
    for t_s, v_s in reversed(pts[:-1]):
        if (t_end - t_s).total_seconds() >= window_s: return max(0.0, v_end - v_s)
    return max(0.0, pts[-1][1] - pts[0][1])

def promql_last(pts):
    # Instant vector — most recent gauge value
    return pts[-1][1] if pts else 0.0

print("=" * 65)
print("PromQL Query Results")
print("=" * 65)

print("\n📊 Query: rate(http_requests_total[5m])")
for svc in [s[0] for s in SERVICES]:
    lb  = '{' + f'service="{svc}",namespace="{NAMESPACE}"' + '}'
    rps = promql_rate(prom_tsdb[f'http_requests_total{lb}'])
    print(f'  {svc:<22} {"█"*int(rps/30):<26} {rps:>7.1f} rps')

print("\n📊 Query: increase(http_errors_total[1h])")
for svc in [s[0] for s in SERVICES]:
    lb   = '{' + f'service="{svc}",namespace="{NAMESPACE}"' + '}'
    errs = promql_increase(prom_tsdb[f'http_errors_total{lb}'])
    print(f'  {svc:<22} {"▰"*min(int(errs/500),30):<30} {errs:>9,.0f} errors')

print("\n📊 Query: process_resident_memory_bytes")
for svc in [s[0] for s in SERVICES]:
    lb  = '{' + f'service="{svc}",namespace="{NAMESPACE}"' + '}'
    mem = promql_last(prom_tsdb[f'process_resident_memory_bytes{lb}']) / 1024**2
    print(f'  {svc:<22} {"█"*int(mem/50):<20} {mem:>7.0f} MiB')

### Step 5c — Prometheus alerting rules

Prometheus **alert rules** are PromQL expressions with a threshold and duration.
An alert fires when the condition evaluates to `true` for the specified duration.

```yaml
# Prometheus alert rule (AlertManager)
- alert: HighErrorRate
  expr: rate(http_errors_total[5m]) / rate(http_requests_total[5m]) > 0.05
  for: 2m
  labels:
    severity: critical
  annotations:
    summary: High error rate on {{ labels.service }}
```

We evaluate three production-style alert rules against our synthetic data.

In [ ]:
# ── Evaluate Prometheus-style alert rules ─────────────────────────────────────
ALERT_RULES = [
    {'name':'HighErrorRate',   'thr':0.05,          'sev':'critical', 'desc':'Error rate > 5%'},
    {'name':'HighP99Latency',  'thr':1.0,           'sev':'warning',  'desc':'P99 > 1000ms'},
    {'name':'HighMemoryUsage', 'thr':800*1024**2,   'sev':'warning',  'desc':'RSS > 800MiB'},
]

print("🔔 Alert Rule Evaluation")
print("=" * 65)
firing = []
for rule in ALERT_RULES:
    print(f'\n  [{rule["sev"].upper()}] {rule["name"]} — {rule["desc"]}')
    for svc in [s[0] for s in SERVICES]:
        lb = '{' + f'service="{svc}",namespace="{NAMESPACE}"' + '}'
        if rule['name'] == 'HighErrorRate':
            v = promql_rate(prom_tsdb[f'http_errors_total{lb}']) / max(promql_rate(prom_tsdb[f'http_requests_total{lb}']), 0.001)
            fmt = f'{v*100:.2f}%'
        elif rule['name'] == 'HighP99Latency':
            v = promql_last(prom_tsdb[f'http_request_duration_p99_s{lb}'])
            fmt = f'{v:.3f}s'
        else:
            v = promql_last(prom_tsdb[f'process_resident_memory_bytes{lb}'])
            fmt = f'{v/1024**2:.0f}MiB'
        fires = v > rule['thr']
        print(f'    {"🔴 FIRING" if fires else "🟢 OK    "}  {svc:<22} {fmt}')
        if fires: firing.append(f'[{rule["sev"].upper()}] {rule["name"]} on {svc}')

print(f'\n\n🚨 Total firing alerts: {len(firing)}')
for a in firing: print(f'   {a}')

---
## Phase 6 — Grafana Dashboard Design 📈

### 📖 Concept: What Is Grafana?

> **Grafana** is the industry-standard **visualisation layer** for observability.
> It connects to Prometheus (and many other data sources) and renders time-series
> data into interactive, configurable dashboards.

### Grafana Panel Types

| Panel Type | Best for | PromQL pattern |
|-----------|---------|----------------|
| **Time series** | Trends over time | `rate(metric[5m])` |
| **Stat** | Single current value | `metric` (instant) |
| **Bar gauge** | Ranking across instances | `sum by (service) (...)` |
| **Heatmap** | Distribution over time | `histogram_quantile(0.99, ...)` |
| **Table** | Multi-metric summary | Multiple instant queries |

### Grafana Threshold Colours

Every panel has threshold lines that change colour when a metric crosses a boundary:
🟢 **Green** — nominal  |  🟡 **Yellow** — investigate  |  🔴 **Red** — SLO breached

### Step 6a — Build a Grafana-style 4-panel dashboard

We build a Python visualisation that replicates a real Grafana dashboard —
four panels covering the most common Kubernetes monitoring use cases:

- **Panel 1:** Time Series — Request Rate
- **Panel 2:** Bar Gauge — P99 Latency vs SLO
- **Panel 3:** Heatmap — Error Rate over Time
- **Panel 4:** Stat Table — Current KPIs with status icons

In [ ]:
# ── Grafana-style 4-panel dashboard ──────────────────────────────────────────
fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.32)
fig.suptitle('📈 Grafana Dashboard — Media Streaming Platform  |  Last 60 minutes',
             fontsize=13, fontweight='bold')

PANEL_COLORS = {'cdn-gateway':'#1f77b4','stream-service':'#17becf','auth-service':'#2ca02c',
                'recommend-svc':'#d62728','encoding-service':'#9467bd','ml-engine':'#ff7f0e',
                'storage-service':'#8c564b','token-db':'#bcbd22','metrics-db':'#7f7f7f'}

# Panel 1: Time Series — RPS
ax1 = fig.add_subplot(gs[0, 0])
for svc in [s[0] for s in SERVICES[:5]]:
    d = df_metrics[df_metrics['service']==svc]
    ax1.plot(d['t_min'], d['http_requests_per_second'], label=svc, color=PANEL_COLORS[svc], lw=1.5)
ax1.set_title('① Time Series — Request Rate (RPS)', fontweight='bold')
ax1.set_ylabel('Requests / sec')
ax1.set_xlabel('Time (min)')
ax1.legend(fontsize=7)

# Panel 2: Bar Gauge — P99 vs SLO
ax2 = fig.add_subplot(gs[0, 1])
SLO_MS = 500
lp99 = df_metrics.groupby('service')['http_request_duration_p99_ms'].last().sort_values()
bcolors = ['#d62728' if v>SLO_MS else '#FF9800' if v>SLO_MS*0.7 else '#4CAF50' for v in lp99.values]
bars2 = ax2.barh(range(len(lp99)), lp99.values, color=bcolors)
ax2.set_yticks(range(len(lp99)))
ax2.set_yticklabels(lp99.index, fontsize=8)
ax2.axvline(SLO_MS, color='red', linestyle='--', lw=1.5, label=f'SLO={SLO_MS}ms')
ax2.set_title('② Bar Gauge — P99 Latency vs SLO', fontweight='bold')
ax2.set_xlabel('P99 Latency (ms)')
ax2.legend(fontsize=8)

# Panel 3: Heatmap — Error Rate
ax3 = fig.add_subplot(gs[1, 0])
hsvcs = [s[0] for s in SERVICES]
NB = 20
bsz = DURATION_MINS/NB
hm  = np.zeros((len(hsvcs), NB))
for i, svc in enumerate(hsvcs):
    d = df_metrics[df_metrics['service']==svc]
    for j in range(NB):
        w = d[(d['t_min']>=j*bsz)&(d['t_min']<(j+1)*bsz)]
        hm[i,j] = w['error_rate'].mean()*100 if len(w)>0 else 0
sns.heatmap(hm, ax=ax3, cmap='YlOrRd',
             xticklabels=[f'{int(j*bsz)}m' if j%4==0 else '' for j in range(NB)],
             yticklabels=hsvcs, cbar_kws={'label':'Error rate (%)','shrink':0.8}, linewidths=0.3)
ax3.set_title('③ Heatmap — Error Rate % Over Time', fontweight='bold')
ax3.tick_params(axis='y', labelsize=7, rotation=0)
ax3.tick_params(axis='x', labelsize=7)

# Panel 4: KPI Table
ax4 = fig.add_subplot(gs[1, 1])
ax4.axis('off')
kpis = []
for svc in [s[0] for s in SERVICES]:
    d   = df_metrics[df_metrics['service']==svc]
    err = d['error_rate'].iloc[-1]
    kpis.append([svc, f'{d["http_requests_per_second"].iloc[-1]:.0f}',
                 f'{d["http_request_duration_p99_ms"].iloc[-1]:.0f}ms',
                 f'{err*100:.2f}%',
                 f'{d["process_resident_memory_bytes"].iloc[-1]/1024**2:.0f}MiB',
                 '🔴' if err>0.05 else '🟡' if err>0.01 else '🟢'])
tbl = ax4.table(cellText=kpis,
                colLabels=['Service','RPS','P99','Err%','Mem','Status'],
                cellLoc='center', loc='center', bbox=[0,0,1,1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
for (r,c), cell in tbl.get_celld().items():
    if r==0: cell.set_facecolor('#2d2d44'); cell.set_text_props(color='white',fontweight='bold')
    elif r%2==0: cell.set_facecolor('#f5f5f5')
ax4.set_title('④ Stat Table — Current Service KPIs', fontweight='bold', y=1.02)

plt.savefig('/tmp/lab15_grafana.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafana-style dashboard rendered")

---
## Phase 7 — Kubernetes Health Signals 🏥

### 📖 Concept: What Are Health Signals?

> **Health signals** are specific observability indicators that reliably predict
> or diagnose service degradation. Unlike raw metrics, health signals are
> **pre-interpreted** — each one has a clear operational meaning.

### The Seven Core Kubernetes Health Signals

| Signal | Source | Indication | Alert Threshold |
|--------|--------|------------|----------------|
| **Pod Restart Count** | K8s events | Crash loop, OOM, probe failure | > 3/hour |
| **OOMKill Events** | K8s events (`OOMKilling`) | Memory limit too low or leak | Any |
| **CrashLoopBackOff** | K8s events | Persistent container failure | Any |
| **Probe Failures** | K8s events (`Unhealthy`) | App not ready or alive | > 2 consecutive |
| **CPU Throttling** | cAdvisor metrics | CPU limit too restrictive | > 25% throttling |
| **Memory Saturation** | Prometheus gauge | Approaching memory limit | > 80% of limit |
| **Error Rate** | App metrics | Direct SLO violation | > 5% of requests |

### Step 7a — Extract health signals from all three pillars

We mine each observability pillar for its health signals and compile them
into a unified signal table.

This is exactly what an AIOps health-scoring engine does — it **fuses signals**
from disparate sources into a single operational risk view.

In [ ]:
# ── Extract health signals from all three pillars ──────────────────────────────
health_signals = []

# Signal 1: Error Rate (Metrics)
for svc in [s[0] for s in SERVICES]:
    d    = df_metrics[df_metrics['service']==svc]
    lerr = d['error_rate'].iloc[-1]
    perr = d['error_rate'].max()
    sev  = 'CRITICAL' if perr>0.10 else 'WARNING' if perr>0.05 else 'OK'
    health_signals.append({'service':svc,'signal':'ErrorRate','value':round(lerr*100,3),
                            'unit':'%','severity':sev,'pillar':'Metrics'})

# Signal 2: P99 Latency (Metrics)
SLO_LAT = 500
for svc in [s[0] for s in SERVICES]:
    d   = df_metrics[df_metrics['service']==svc]
    p99 = d['http_request_duration_p99_ms'].iloc[-1]
    sev = 'CRITICAL' if p99>SLO_LAT*2 else 'WARNING' if p99>SLO_LAT else 'OK'
    health_signals.append({'service':svc,'signal':'P99Latency','value':round(p99,1),
                            'unit':'ms','severity':sev,'pillar':'Metrics'})

# Signal 3: Memory Saturation (Metrics)
MEM_LIMITS = {'cdn-gateway':128,'stream-service':256,'auth-service':128,
              'recommend-svc':512,'encoding-service':1024,'ml-engine':1024,
              'storage-service':256,'token-db':256,'metrics-db':256}
for svc in [s[0] for s in SERVICES]:
    d   = df_metrics[df_metrics['service']==svc]
    mem = d['process_resident_memory_bytes'].iloc[-1]/1024**2
    sat = mem/MEM_LIMITS.get(svc,256)*100
    sev = 'CRITICAL' if sat>90 else 'WARNING' if sat>75 else 'OK'
    health_signals.append({'service':svc,'signal':'MemSaturation','value':round(sat,1),
                            'unit':'%','severity':sev,'pillar':'Metrics'})

# Signal 4: OOMKill Events (K8s Events)
oom_df = df_events[df_events['reason']=='OOMKilling']
for svc in [s[0] for s in SERVICES]:
    cnt = len(oom_df[oom_df['service']==svc])
    health_signals.append({'service':svc,'signal':'OOMKillCount','value':cnt,
                            'unit':'events','severity':'CRITICAL' if cnt>0 else 'OK','pillar':'Events'})

# Signal 5: CrashLoop/BackOff Events (K8s Events)
crash_df = df_events[df_events['reason'].isin(['CrashLoopBackOff','BackOff'])]
for svc in [s[0] for s in SERVICES]:
    cnt = len(crash_df[crash_df['service']==svc])
    sev = 'CRITICAL' if cnt>=2 else 'WARNING' if cnt==1 else 'OK'
    health_signals.append({'service':svc,'signal':'CrashRestarts','value':cnt,
                            'unit':'events','severity':sev,'pillar':'Events'})

# Signal 6: Log Error Rate (Logs)
for svc in [s[0] for s in SERVICES]:
    sl = df_logs[df_logs['service']==svc]
    if len(sl)==0: continue
    pct = len(sl[sl['level']=='ERROR'])/len(sl)*100
    sev = 'CRITICAL' if pct>20 else 'WARNING' if pct>10 else 'OK'
    health_signals.append({'service':svc,'signal':'LogErrorRate','value':round(pct,2),
                            'unit':'%','severity':sev,'pillar':'Logs'})

df_signals = pd.DataFrame(health_signals)
print(f'✅ Extracted {len(df_signals)} health signal readings')
for sev, cnt in df_signals['severity'].value_counts().items():
    icon = '🔴' if sev=='CRITICAL' else '🟡' if sev=='WARNING' else '🟢'
    print(f'  {icon}  {sev:<10} {cnt} readings')
print('\nAll CRITICAL signals:')
crit = df_signals[df_signals['severity']=='CRITICAL'][['service','signal','value','unit','pillar']]
print(tabulate(crit.sort_values('service'), headers='keys', tablefmt='rounded_outline', showindex=False))

### Step 7b — Health signal heatmap

The heatmap shows every signal (rows) × every service (columns).
Red = critical, orange = warning, green = OK.

A **column that is mostly red** signals a service in serious trouble.
Notice `ml-engine` and `recommend-svc` — they were the degraded services in our scenario.

In [ ]:
# ── Health signal heatmap ─────────────────────────────────────────────────────
SEV_NUM = {'OK':0,'WARNING':1,'CRITICAL':2}
signals = df_signals['signal'].unique()
svcs    = [s[0] for s in SERVICES]

matrix = pd.DataFrame(index=signals, columns=svcs, dtype=float)
for _, row in df_signals.iterrows():
    matrix.loc[row['signal'], row['service']] = SEV_NUM[row['severity']]
matrix = matrix.fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(matrix.astype(float), ax=ax,
             cmap=['#4CAF50','#FF9800','#F44336'], vmin=0, vmax=2,
             linewidths=0.5, linecolor='#cccccc', annot=True, fmt='.0f',
             cbar_kws={'ticks':[0.33,1.0,1.67],'label':'0=OK  1=WARN  2=CRIT'})
ax.set_title('🏥 Kubernetes Health Signal Matrix — All Services × All Signals', fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
for i, sig in enumerate(signals):
    pillar = df_signals[df_signals['signal']==sig]['pillar'].iloc[0]
    icon   = {'Metrics':'📊','Events':'⚡','Logs':'📋'}.get(pillar,'')
    ax.text(-0.3, i+0.5, icon, ha='center', va='center', fontsize=9, transform=ax.transData)
plt.tight_layout()
plt.savefig('/tmp/lab15_health_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Health signal heatmap rendered")

---
## Phase 8 — Multi-Signal Health Scoring 🎯

### 📖 Concept: Composite Health Scoring

> Individual health signals are noisy. A single elevated latency reading might be
> a statistical outlier. **Composite health scoring** combines multiple signals
> into a single number that is far more reliable than any individual signal alone.

### Scoring Methodology

```
health_score = Σ  weight_i × severity_value_i

severity_value:  OK=0   WARNING=1   CRITICAL=3   (non-linear: critical = 3× warning)
```

**Signal weights** reflect operational importance:

| Signal | Weight | Rationale |
|--------|--------|-----------|
| OOMKillCount | 0.25 | Definitive evidence of resource overrun |
| CrashRestarts | 0.20 | Persistent failure needing immediate action |
| ErrorRate | 0.20 | Direct user-facing impact |
| P99Latency | 0.15 | SLO violation |
| MemSaturation | 0.10 | Leading indicator of future OOMKill |
| LogErrorRate | 0.10 | Corroborating evidence from logs |

Final score normalised to **0.0** (healthy) → **1.0** (maximally degraded).

In [ ]:
# ── Compute composite health score ───────────────────────────────────────────
SIGNAL_WEIGHTS = {'OOMKillCount':0.25,'CrashRestarts':0.20,'ErrorRate':0.20,
                   'P99Latency':0.15,'MemSaturation':0.10,'LogErrorRate':0.10}
SEV_SCORE = {'OK':0.0,'WARNING':1.0,'CRITICAL':3.0}

health_scores = {}
for svc in svcs:
    ss = df_signals[df_signals['service']==svc]
    w_sum = t_w = 0.0
    for _, row in ss.iterrows():
        w = SIGNAL_WEIGHTS.get(row['signal'], 0.05)
        w_sum += w * SEV_SCORE[row['severity']]
        t_w   += w
    raw   = w_sum / max(t_w, 0.001)
    score = min(raw / 3.0, 1.0)
    grade = 'F' if raw>2.0 else 'D' if raw>1.0 else 'C' if raw>0.5 else 'B' if raw>0.2 else 'A'
    health_scores[svc] = {'health_score':round(score,4),'raw_score':round(raw,4),'grade':grade}

df_health = pd.DataFrame(health_scores).T.sort_values('health_score', ascending=False)
df_health.index.name = 'service'
print("🏆 Service Health Scores (worst → best)")
print("=" * 55)
print(tabulate(df_health[['health_score','grade','raw_score']], headers='keys', tablefmt='rounded_outline'))
print("\n🔴 Services requiring immediate attention:")
for svc, row in df_health[df_health['grade'].isin(['F','D'])].iterrows():
    print(f'  {svc:<22} Grade={row["grade"]}  Score={row["health_score"]:.3f}')

### Step 8b — Consolidated health dashboard

The final dashboard integrates all three observability pillars into one view:

| Panel | Content | Key insight |
|-------|---------|-------------|
| Top-left | Health score ranking | The composite risk view |
| Top-right | Signal contribution breakdown | Which signals drive each score |
| Bottom-left | Error rate time-series | Trend for degraded services |
| Bottom-right | Service topology + health overlay | Spatial view of incident scope |

> 💡 **Compare the Health Score ranking (Panel 1) to the raw error rate chart (Phase 2).**
> `ml-engine` might not have the highest raw error rate, but it tops the health score
> because it also has OOMKill events and CrashLoopBackOff — the most severe K8s signals.

In [ ]:
# ── Consolidated health dashboard ─────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.48, wspace=0.35)
fig.suptitle('🏥 Kubernetes Observability — Consolidated Health Dashboard',
             fontsize=13, fontweight='bold')

def sc(s): return '#d62728' if s>0.60 else '#FF9800' if s>0.25 else '#4CAF50'

# Panel 1: Health Score Ranking
ax1 = fig.add_subplot(gs[0,0])
sh  = df_health.sort_values('health_score', ascending=True)
brs = ax1.barh(range(len(sh)), sh['health_score'], color=[sc(s) for s in sh['health_score']])
ax1.set_yticks(range(len(sh)))
ax1.set_yticklabels(sh.index, fontsize=8)
ax1.axvline(0.60, color='red',    ls='--', lw=1, alpha=0.7, label='Critical threshold')
ax1.axvline(0.25, color='orange', ls='--', lw=1, alpha=0.7, label='Warning threshold')
ax1.set_title('① Service Health Ranking', fontweight='bold')
ax1.set_xlabel('Health Score (0=healthy, 1=critical)')
ax1.legend(fontsize=7)
for bar, (svc, row) in zip(brs, sh.iterrows()):
    ax1.text(row['health_score']+0.01, bar.get_y()+bar.get_height()/2,
             f'{row["health_score"]:.2f} [{row["grade"]}]', va='center', fontsize=7)

# Panel 2: Signal Contribution
ax2 = fig.add_subplot(gs[0,1])
sigs_p = ['OOMKillCount','CrashRestarts','ErrorRate','P99Latency','MemSaturation','LogErrorRate']
cols_p = ['#d62728','#ff7f0e','#e377c2','#9467bd','#2196f3','#4CAF50']
bot2   = np.zeros(len(svcs))
for sig, col in zip(sigs_p, cols_p):
    vals = [SIGNAL_WEIGHTS.get(sig,0)*SEV_SCORE[
              df_signals[(df_signals['service']==s)&(df_signals['signal']==sig)]['severity'].iloc[0]
              if len(df_signals[(df_signals['service']==s)&(df_signals['signal']==sig)])>0 else 0]
              if len(df_signals[(df_signals['service']==s)&(df_signals['signal']==sig)])>0 else 0
            for s in svcs]
    ax2.bar(range(len(svcs)), vals, bottom=bot2, color=col, label=sig, alpha=0.85)
    bot2 += np.array(vals)
ax2.set_xticks(range(len(svcs)))
ax2.set_xticklabels(svcs, rotation=35, ha='right', fontsize=7)
ax2.set_title('② Signal Contribution Breakdown', fontweight='bold')
ax2.legend(fontsize=6, ncol=2)

# Panel 3: Error Rate Trend
ax3 = fig.add_subplot(gs[1,0])
for svc in svcs:
    d  = df_metrics[df_metrics['service']==svc]
    lw = 2.5 if df_health.loc[svc,'grade'] in ['F','D'] else 0.8
    ax3.plot(d['t_min'], d['error_rate']*100, color=sc(df_health.loc[svc,'health_score']), lw=lw,
              label=svc if lw>1 else None)
ax3.axhline(5, color='red', ls='--', lw=1, alpha=0.6, label='SLO 5%')
ax3.set_xlabel('Time (min)')
ax3.set_ylabel('Error Rate (%)')
ax3.set_title('③ Error Rate — Degraded Services Highlighted', fontweight='bold')
ax3.legend(fontsize=7, ncol=2)

# Panel 4: Topology + Health Overlay
ax4 = fig.add_subplot(gs[1,1])
ax4.set_facecolor('#1a1a2e')
G_t = nx.DiGraph()
for caller, callees in DEPENDENCY_GRAPH.items():
    G_t.add_node(caller)
    for callee in callees: G_t.add_edge(caller, callee)
tpos = {'cdn-gateway':(0.0,0.5),'stream-service':(0.3,0.85),'auth-service':(0.3,0.50),
        'recommend-svc':(0.3,0.15),'encoding-service':(0.6,0.85),'token-db':(0.6,0.50),
        'ml-engine':(0.6,0.20),'storage-service':(0.9,0.85),'metrics-db':(0.9,0.20)}
nc = [sc(df_health.loc[n,'health_score']) if n in df_health.index else '#aaaaaa' for n in G_t.nodes()]
nx.draw_networkx_edges(G_t, tpos, ax=ax4, edge_color='#888888', arrows=True, arrowsize=12, width=1.2)
nx.draw_networkx_nodes(G_t, tpos, ax=ax4, node_color=nc, node_size=900, alpha=0.9)
nx.draw_networkx_labels(G_t, tpos, labels={n:n.replace('-','\n') for n in G_t.nodes()},
                         ax=ax4, font_size=5.5, font_color='white', font_weight='bold')
ax4.set_title('④ Topology — Health Overlay', color='white', fontweight='bold')
ax4.axis('off')
lhd = [mpatches.Patch(color='#d62728',label='Critical'),
        mpatches.Patch(color='#FF9800',label='Warning'),
        mpatches.Patch(color='#4CAF50',label='Healthy')]
ax4.legend(handles=lhd, fontsize=7, facecolor='#2d2d44', labelcolor='white', loc='lower right')

plt.savefig('/tmp/lab15_health_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Consolidated health dashboard rendered")
print("\n" + "="*55)
print("  FINAL HEALTH SUMMARY")
print("="*55)
for svc, row in df_health.sort_values('health_score', ascending=False).iterrows():
    icon = '🔴' if row['grade'] in ['F','D'] else '🟡' if row['grade']=='C' else '🟢'
    print(f'  {icon}  {svc:<22} Grade={row["grade"]}  Score={row["health_score"]:.3f}')

---
## Phase 9 — Cleanup 🧹

Deleting the namespace removes all pods, deployments, and services.

> ⚠️ Only run this when you have finished exploring the dashboards.

In [ ]:
# ── Delete the lab namespace ──────────────────────────────────────────────────
print(f'Deleting namespace "{NAMESPACE}"...')
out, rc = run(f'kubectl delete namespace {NAMESPACE} --wait=false')
if rc == 0:
    print(f'\n✅ Namespace "{NAMESPACE}" deletion initiated.')
    print(f'   Verify: kubectl get namespace {NAMESPACE}')
else:
    print('\n⚠️  Could not delete — it may already be gone.')

---
## 📚 Lab Recap & Key Takeaways

### The Three Pillars — Summarised

```
 ┌──────────────┬────────────────────────────────────────┬──────────────┐
 │   PILLAR     │  What it provides                      │  Tool stack  │
 ├──────────────┼────────────────────────────────────────┼──────────────┤
 │ 📊 Metrics   │ Numeric time-series, alerting, trending│  Prometheus  │
 │ 📋 Logs      │ Rich context, error details, trace IDs │  Loki / ELK  │
 │ ⚡ Events    │ K8s state changes, ground truth for RCA│  kubectl     │
 └──────────────┴────────────────────────────────────────┴──────────────┘
```

No single pillar is sufficient — production observability needs **all three**.

---

### 🏆 Key Concepts Mastered

| Topic | What You Learned |
|-------|------------------|
| **Prometheus Data Model** | Metric names, labels, time series, TSDB structure |
| **Metric Types** | Counter, Gauge, Histogram, Summary — when to use each |
| **PromQL** | `rate()`, `increase()`, `sum by()`, alert rule expressions |
| **Grafana Panels** | Time series, bar gauge, heatmap, stat table, threshold colours |
| **K8s Events** | `OOMKilling`, `CrashLoopBackOff`, `Unhealthy` — event taxonomy |
| **Health Signals** | Seven core signals across all three observability pillars |
| **Health Scoring** | Weighted composite scoring to rank services by operational risk |

---

### 💡 The Observability Maturity Ladder

```
Level 1 — Reactive    │  Found out from users         │  No observability
Level 2 — Proactive   │  Alerts fire first            │  Metrics + basic alerts
Level 3 — Predictive  │  Spot it before it hurts      │  All 3 pillars + trending
Level 4 — AIOps       │  System self-diagnoses        │  Multi-signal health scoring
```

This lab implements **Level 4** — the health scorer in Phase 8 fuses
all three pillars to automatically rank services by operational risk.

---

### 🔗 Session 15 Topic Coverage

| Session Topic | Lab Phase |
|---------------|-----------|
| Metrics, Logs, Events in Kubernetes | Phases 2, 3, 4 |
| Prometheus Concepts | Phase 5 — data model, PromQL, alert rules |
| Grafana Concepts | Phase 6 — panel types, thresholds, dashboards |
| Kubernetes Health Signals | Phase 7 — seven core signals + heatmap |
| Multi-signal synthesis | Phase 8 — composite health scoring |

---

*Lab 15 complete — well done! 🎉*